# Notebook 02c — bfast unificado sobre las 4 estaciones de manglar real

Re-ejecuta la detección de quiebres estructurales con bfast restringiéndola a las cuatro estaciones que efectivamente miden cobertura de manglar denso ---Caño Palos, Caño Clarín, CP Aguas Negras y CP Luna--- y excluye las cuatro estaciones limnológicas ---Isla Boquerón, Punta Cerro, Punta Chino y Río Sevilla--- que están sobre la lámina de agua del complejo lagunar central. La motivación es eliminar el ruido espectral de las estaciones que monitorean agua y refinar la detección de quiebres asociados al evento La Niña 2020--2021, en la medida en que el análisis previo del notebook `02b_bfast_ndvi.R.ipynb` sobre las 8 estaciones podía diluir la señal específica del manglar al promediar comportamientos espectralmente heterogéneos.

**Insumos:** `outputs/tables/serie_temporal_ndvi_definitiva.csv`

**Productos:**
- `outputs/tables/bfast_manglar_unificado.csv`
- `outputs/figures/bfast_manglar_unif_<estacion>_h<XX>.png` (8 figuras)

In [ ]:
suppressPackageStartupMessages({
  library(bfast)
  library(dplyr)
  library(tidyr)
  library(readr)
  library(zoo)
})

setwd('/home/rstudio/work/proyecto-cgsm')
csv_in  <- 'outputs/tables/serie_temporal_ndvi_definitiva.csv'
out_csv <- 'outputs/tables/bfast_manglar_unificado.csv'
out_fig <- 'outputs/figures'
dir.create(out_fig, showWarnings = FALSE, recursive = TRUE)

manglar_4 <- c('Cano_Palos', 'Cano_Clarin', 'CP_Aguas_Negras', 'CP_Luna')
cat('Estaciones de manglar seleccionadas:\n')
print(manglar_4)

## 1. Cargar serie temporal y filtrar a las 4 estaciones de manglar

In [ ]:
df <- read_csv(csv_in, show_col_types = FALSE)
cat('Estaciones disponibles en el CSV:\n')
print(table(df$subzona))

df_man <- df %>%
  filter(subzona %in% manglar_4) %>%
  mutate(date = as.Date(date)) %>%
  arrange(subzona, date)

cat('\nObservaciones por estación de manglar:\n')
print(table(df_man$subzona))
cat('\nRango temporal:\n')
print(range(df_man$date))

## 2. Aplicar bfast con h = 0,15 y h = 0,10 a cada estación

Las series se completan con interpolación lineal en los meses sin observación para garantizar continuidad temporal mensual, requisito de bfast. El parámetro `h` controla la ventana mínima entre quiebres como fracción de la serie: 0,15 es el valor del análisis previo y 0,10 ofrece mayor sensibilidad para detectar eventos más cercanos.

In [ ]:
resultados <- list()

for (est in manglar_4) {
  cat('\n=== ', est, ' ===\n', sep = '')
  serie <- df_man %>% filter(subzona == est)

  if (nrow(serie) < 36) {
    cat('  Insuficientes observaciones (', nrow(serie), '), salto.\n', sep = '')
    next
  }

  fechas_mes <- seq(min(serie$date), max(serie$date), by = 'month')
  serie_mes  <- data.frame(date = fechas_mes) %>%
    left_join(serie %>% select(date, ndvi), by = 'date')

  ts_obj <- ts(serie_mes$ndvi,
               start = c(as.integer(format(min(fechas_mes), '%Y')),
                         as.integer(format(min(fechas_mes), '%m'))),
               frequency = 12)

  if (any(is.na(ts_obj))) ts_obj <- na.approx(ts_obj, rule = 2)

  for (h_val in c(0.15, 0.10)) {
    tryCatch({
      fit <- bfast(ts_obj, h = h_val, season = 'harmonic', max.iter = 5)
      bp_idx <- fit$output[[length(fit$output)]]$Vt.bp
      if (length(bp_idx) > 0 && !any(is.na(bp_idx))) {
        fechas_brks <- time(ts_obj)[bp_idx]
        fechas_brks_str <- paste(round(fechas_brks, 2), collapse = '; ')
        n_brks <- length(bp_idx)
      } else {
        n_brks <- 0
        fechas_brks_str <- ''
      }
      resultados[[length(resultados) + 1]] <- data.frame(
        estacion = est, h = h_val, n_quiebres = n_brks,
        fechas_quiebres = fechas_brks_str,
        n_obs = length(ts_obj),
        rango = paste(min(fechas_mes), max(fechas_mes), sep = ' a '),
        stringsAsFactors = FALSE
      )
      cat('  h = ', h_val, ': ', n_brks, ' quiebres  ', fechas_brks_str, '\n', sep = '')

      # Guardar gráfico
      png(file.path(out_fig,
          sprintf('bfast_manglar_unif_%s_h%02d.png', est, round(h_val * 100))),
          width = 1000, height = 700)
      plot(fit, main = sprintf('%s — bfast h=%.2f', est, h_val))
      dev.off()
    }, error = function(e) {
      cat('  h = ', h_val, ': ERROR — ', e$message, '\n', sep = '')
      resultados[[length(resultados) + 1]] <<- data.frame(
        estacion = est, h = h_val, n_quiebres = NA,
        fechas_quiebres = paste('error:', e$message),
        n_obs = length(ts_obj),
        rango = paste(min(fechas_mes), max(fechas_mes), sep = ' a '),
        stringsAsFactors = FALSE
      )
    })
  }
}

df_res <- bind_rows(resultados)
write_csv(df_res, out_csv)

cat('\n=== RESUMEN bfast 4 estaciones manglar (unificadas) ===\n')
print(df_res)
cat('\nGuardado: ', out_csv, '\n', sep = '')
cat('Gráficos PNG: ', out_fig, '/bfast_manglar_unif_*.png\n', sep = '')

## 3. Interpretación esperada

El análisis bfast previo sobre las 8 estaciones identificó un quiebre estructural generalizado en 2016 ---atribuido al El Niño 2015-2016--- pero no detectó de manera consistente el evento La Niña 2020-2021 sobre la mayoría de las estaciones. La hipótesis es que al promediar señales espectrales heterogéneas (manglar denso + lámina de agua) la respuesta del dosel se diluye y bfast no encuentra suficiente discontinuidad estructural para reportarla como quiebre. Al restringir el análisis a las cuatro estaciones que sí miden manglar denso, el evento 2020 debería emerger con mayor claridad, particularmente en CP Aguas Negras donde INVEMAR documenta una pérdida del 33 % del arbolado por inundación permanente.